# 🥇 Gold Layer - Customer 360 View
## AWS S3 External Storage via Unity Catalog

**Purpose**: Create comprehensive customer 360 view with all customer interactions

**Source**: `workspace.`workspace-adventureworks-silver`` (S3)
**Target**: `workspace.`workspace-adventureworks-gold`` (S3)

**Gold Tables to Create**:
1. **gold_customer_360** - Complete customer profile with metrics
2. **gold_customer_segmentation** - RFM-based customer segments
3. **gold_customer_lifetime_value** - Customer lifetime value analysis

**Business Metrics**:
- Recency, Frequency, Monetary (RFM) analysis
- Customer lifetime value (CLV)
- Customer acquisition and churn indicators
- Geographic distribution
- Purchase patterns and preferences

In [0]:
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
import time

# Configuration
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"
GOLD_SCHEMA = "workspace.`workspace-adventureworks-gold`"

print("=" * 80)
print("🥇 GOLD LAYER - CUSTOMER 360 VIEW")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SILVER_SCHEMA} (AWS S3)")
print(f"Target: {GOLD_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Customer Segmentation using RFM Analysis
import pyspark.sql.functions as F

print("\n" + "=" * 80)
print("🎯 BUILDING gold_customer_segmentation")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"{SILVER_SCHEMA}.fact_sales")
    dim_customer = spark.table(f"{SILVER_SCHEMA}.dim_customer").filter(col("is_current") == True)
    
    reference_date = current_date()
    
    # Aggregate to order level first (fact_sales has line-level detail)
    order_aggregates = (fact_sales
        .groupBy("sales_order_id", "customer_id", "order_date")
        .agg(F.sum("line_total").alias("order_total"))
    )
    
    # Calculate RFM per customer
    rfm_data = (order_aggregates
        .groupBy("customer_id")
        .agg(
            F.max("order_date").alias("last_purchase_date"),
            F.count("sales_order_id").alias("frequency"),
            F.sum("order_total").alias("monetary")
        )
        .withColumn("recency", datediff(lit(reference_date), col("last_purchase_date")))
    )
    
    # Calculate RFM scores
    rfm_scored = (rfm_data
        .withColumn("recency_score", ntile(5).over(Window.orderBy(col("recency").asc())))
        .withColumn("frequency_score", ntile(5).over(Window.orderBy(col("frequency").desc())))
        .withColumn("monetary_score", ntile(5).over(Window.orderBy(col("monetary").desc())))
        .withColumn("rfm_score", col("recency_score") + col("frequency_score") + col("monetary_score"))
    )
    
    # Assign segments
    gold_customer_segmentation = (rfm_scored
        .join(dim_customer, "customer_id")  # Use column name for equi-join to avoid ambiguity
        .select(
            col("customer_id"),
            col("account_number"),
            col("recency"),
            col("frequency"),
            col("monetary"),
            col("recency_score"),
            col("frequency_score"),
            col("monetary_score"),
            col("rfm_score")
        )
        .withColumn("customer_segment",
            when((col("recency_score") >= 4) & (col("frequency_score") >= 4) & (col("monetary_score") >= 4), "Champions")
            .when((col("recency_score") >= 3) & (col("frequency_score") >= 3) & (col("monetary_score") >= 3), "Loyal Customers")
            .when((col("recency_score") >= 4) & (col("frequency_score") <= 2), "New Customers")
            .when((col("recency_score") <= 2) & (col("frequency_score") >= 4), "At Risk")
            .when((col("recency_score") <= 2) & (col("frequency_score") <= 2) & (col("monetary_score") >= 4), "Cant Lose Them")
            .when((col("recency_score") <= 2) & (col("frequency_score") <= 2) & (col("monetary_score") <= 2), "Hibernating")
            .when((col("recency_score") >= 3) & (col("frequency_score") <= 2) & (col("monetary_score") <= 2), "Promising")
            .otherwise("Needs Attention")
        )
        .withColumn("segment_priority",
            when(col("customer_segment") == "Champions", 1)
            .when(col("customer_segment") == "Loyal Customers", 2)
            .when(col("customer_segment") == "Cant Lose Them", 3)
            .when(col("customer_segment") == "At Risk", 4)
            .when(col("customer_segment") == "Promising", 5)
            .when(col("customer_segment") == "New Customers", 6)
            .when(col("customer_segment") == "Needs Attention", 7)
            .otherwise(8)
        )
    )
    
    # Write to Gold
    gold_customer_segmentation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD_SCHEMA}.gold_customer_segmentation")
    
    duration = time.time() - start_time
    row_count = gold_customer_segmentation.count()
    
    transformation_results.append({"table": "gold_customer_segmentation", "status": "success", "row_count": row_count, "duration_seconds": round(duration, 2)})
    
    print(f"\n✅ gold_customer_segmentation created: {row_count:,} rows in {duration:.2f}s")
    segment_dist = gold_customer_segmentation.groupBy("customer_segment").count().orderBy("count", ascending=False)
    print("\n📊 Customer Segment Distribution:")
    display(segment_dist)
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_customer_segmentation", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Customer Lifetime Value Analysis
import pyspark.sql.functions as F

print("\n" + "=" * 80)
print("💵 BUILDING gold_customer_lifetime_value")
print("=" * 80)

start_time = time.time()

try:
    # Read gold_sales_by_customer (aggregated by year)
    gold_sales = spark.table(f"{GOLD_SCHEMA}.gold_sales_by_customer")
    
    # Aggregate across all years to get lifetime metrics
    customer_lifetime = (gold_sales
        .groupBy("customer_id", "account_number")
        .agg(
            F.sum("total_orders").alias("lifetime_orders"),
            F.sum("total_revenue").alias("lifetime_revenue"),
            F.min("first_order_date").alias("first_order_date"),
            F.max("last_order_date").alias("last_order_date")
        )
    )
    
    # Calculate CLV metrics
    gold_clv = (customer_lifetime
        .withColumn("customer_tenure_days",
            datediff(col("last_order_date"), col("first_order_date"))
        )
        .withColumn("days_since_last_order",
            datediff(current_date(), col("last_order_date"))
        )
        .withColumn("avg_order_value",
            when(col("lifetime_orders") > 0,
                col("lifetime_revenue") / col("lifetime_orders")
            ).otherwise(0)
        )
        .withColumn("annual_revenue",
            when(col("customer_tenure_days") > 0,
                col("lifetime_revenue") / col("customer_tenure_days") * 365
            ).otherwise(col("lifetime_revenue"))
        )
        .withColumn("estimated_clv_3year",
            col("annual_revenue") * 3 * 0.85  # 3 years with 15% discount for churn
        )
        .withColumn("customer_status",
            when(col("days_since_last_order") <= 90, "Active")
            .when(col("days_since_last_order") <= 180, "At Risk")
            .when(col("days_since_last_order") <= 365, "Dormant")
            .otherwise("Inactive")
        )
        .withColumn("clv_tier",
            when(col("estimated_clv_3year") >= 10000, "Platinum")
            .when(col("estimated_clv_3year") >= 5000, "Gold")
            .when(col("estimated_clv_3year") >= 2000, "Silver")
            .otherwise("Bronze")
        )
    )
    
    # Add value ranking
    gold_clv = gold_clv.withColumn(
        "clv_rank",
        dense_rank().over(Window.orderBy(col("estimated_clv_3year").desc()))
    )
    
    # Write to Gold
    (gold_clv
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_customer_lifetime_value")
    )
    
    duration = time.time() - start_time
    row_count = gold_clv.count()
    
    transformation_results.append({
        "table": "gold_customer_lifetime_value",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_customer_lifetime_value created: {row_count:,} rows in {duration:.2f}s")
    
    # Show CLV tier distribution
    clv_dist = gold_clv.groupBy("clv_tier").agg(
        F.count("*").alias("customer_count"),
        F.sum("estimated_clv_3year").alias("total_clv")
    ).orderBy("total_clv", ascending=False)
    print("\n💰 CLV Tier Distribution:")
    display(clv_dist)
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_customer_lifetime_value", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 CUSTOMER 360 ANALYTICS SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nGold Tables Created: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ GOLD CUSTOMER 360 COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ GOLD CUSTOMER 360 COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")